In [15]:
!nvidia-smi

Tue Apr 28 23:57:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             53W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [16]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory GB:", torch.cuda.get_device_properties(0).total_memory / 1e9)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA build: 12.8
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Memory GB: 85.094825984


In [17]:
from pathlib import Path
import os

repo_dir = Path("/content/modded-nanogpt")
repo_url = "https://github.com/bluepeach1121/modded-nanogpt.git"
branch = "modded-gpt-test"

if repo_dir.exists():
    print("Repo already exists. Using existing clone.")
    os.chdir(repo_dir)
    !git fetch origin
    !git checkout {branch}
    !git pull
else:
    print("Repo not found. Cloning...")
    %cd /content
    !git clone {repo_url}
    %cd /content/modded-nanogpt
    !git checkout {branch}

print("\nCurrent branch:")
!git branch

print("\nStatus:")
!git status

Repo already exists. Using existing clone.
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.25 KiB | 641.00 KiB/s, done.
From https://github.com/bluepeach1121/modded-nanogpt
   e156f52..2d3ae10  modded-gpt-test -> origin/modded-gpt-test
Already on 'modded-gpt-test'
Your branch is behind 'origin/modded-gpt-test' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating e156f52..2d3ae10
Fast-forward
 .../track_3_optimization/smoke_train_gpt_simple.py |   2 +-
 track3_colab_test.ipynb                            | 295 ++++-----------------
 2 files changed, 53 insertions(+), 244 deletions(-)

Current branch:
  master
* modded-gpt-test

Status:
On branch modded-gpt-test
Your branch is up to date with 'origin/modded-gpt-test'.

nothing to commit, working tree clean


In [18]:
from pathlib import Path

repo_root = Path.cwd()
script_path = Path("records/track_3_optimization/train_gpt_simple.py")

print("Repo root:", repo_root)
print("records exists:", Path("records").exists())
print("data exists:", Path("data").exists())
print("Script exists:", script_path.exists())
print("Script path:", script_path.resolve())

Repo root: /content/modded-nanogpt
records exists: True
data exists: True
Script exists: True
Script path: /content/modded-nanogpt/records/track_3_optimization/train_gpt_simple.py


In [19]:
from huggingface_hub import hf_hub_download
from pathlib import Path

repo_id = "kjj0/fineweb10B-gpt2"
data_dir = Path("data/fineweb10B")
data_dir.mkdir(parents=True, exist_ok=True)

required_files = ["fineweb_val_000000.bin"]
required_files += [f"fineweb_train_{i:06d}.bin" for i in range(1, 21)]

for fname in required_files:
    local_file = data_dir / fname

    if local_file.exists() and local_file.stat().st_size > 0:
        print(f"Already exists, skipping: {fname}")
        continue

    print(f"Downloading: {fname}")
    hf_hub_download(
        repo_id=repo_id,
        filename=fname,
        local_dir=data_dir,
        repo_type="dataset",
    )

print("\nDownload/check complete.")

# Verify downloaded files
files = sorted(data_dir.glob("*.bin"))

print("\nNumber of .bin files:", len(files))
total_size_gb = 0

for f in files:
    size_gb = f.stat().st_size / 1e9
    total_size_gb += size_gb
    print(f"{f.name}: {size_gb:.3f} GB")

print(f"\nTotal size: {total_size_gb:.3f} GB")

Already exists, skipping: fineweb_val_000000.bin
Already exists, skipping: fineweb_train_000001.bin
Already exists, skipping: fineweb_train_000002.bin
Already exists, skipping: fineweb_train_000003.bin
Already exists, skipping: fineweb_train_000004.bin
Already exists, skipping: fineweb_train_000005.bin
Already exists, skipping: fineweb_train_000006.bin
Already exists, skipping: fineweb_train_000007.bin
Already exists, skipping: fineweb_train_000008.bin
Already exists, skipping: fineweb_train_000009.bin
Already exists, skipping: fineweb_train_000010.bin
Already exists, skipping: fineweb_train_000011.bin
Already exists, skipping: fineweb_train_000012.bin
Already exists, skipping: fineweb_train_000013.bin
Already exists, skipping: fineweb_train_000014.bin
Already exists, skipping: fineweb_train_000015.bin
Already exists, skipping: fineweb_train_000016.bin
Already exists, skipping: fineweb_train_000017.bin
Already exists, skipping: fineweb_train_000018.bin
Already exists, skipping: fineweb

In [ ]:
!torchrun --standalone --nproc_per_node=1 records/track_3_optimization/train_gpt_simple.py

logs/c88e8b84-4305-4de0-8520-92ab786c60d8.txt
step:0/50 val_loss:10.82583 train_time:0.000s step_avg:0.08ms
step:1/50 train_time:5.988s step_avg:5988.06ms
step:2/50 train_time:8.521s step_avg:4260.35ms
step:3/50 train_time:10.988s step_avg:3662.52ms
step:4/50 train_time:13.450s step_avg:3362.51ms
step:5/50 train_time:15.914s step_avg:3182.84ms
step:6/50 train_time:18.379s step_avg:3063.21ms
step:7/50 train_time:20.844s step_avg:2977.68ms
step:8/50 train_time:23.308s step_avg:2913.48ms
step:9/50 train_time:25.775s step_avg:2863.94ms
step:10/50 train_time:28.239s step_avg:2823.95ms
step:11/50 train_time:30.704s step_avg:2791.23ms
step:12/50 train_time:33.167s step_avg:2763.95ms
step:13/50 train_time:35.633s step_avg:2741.02ms
step:14/50 train_time:38.098s step_avg:2721.27ms
step:15/50 train_time:40.562s step_avg:2704.14ms
step:16/50 train_time:43.026s step_avg:2689.15ms
step:17/50 train_time:45.494s step_avg:2676.10ms
step:18/50 train_time:47.958s step_avg:2664.33ms
step:19/50 train_time